# 00 - RDKit'in Projedeki Rol?

Bu notebook, RDKit'i bu projenin ba?lam?nda tan?t?r: **molek?l yap?s?n? Python'da temsil etme** ve kimyasal veriyi ileride makine ??renmesine haz?rlanabilir h?le getirme arac?.

Bu ?al??ma bir bilimsel pipeline de?ildir. Bu notebookta ger?ek veri indirme, final dataset se?imi, modelleme, train/test split, metric/evaluation, Macro F1, benchmark veya bilimsel sonu? ?retimi yap?lmaz.


## ??renme hedefleri

Bu notebookun sonunda ?unlar? ay?rt edebilmelisin:

- RDKit'in ??C NMR ? fonksiyonel grup tahmini projesindeki yard?mc? rol?.
- SMILES metninin RDKit `Mol` nesnesine nas?l d?n??t???n?.
- Descriptor, fingerprint ve SMARTS kavramlar?n?n ne i?e yarad???n?.
- Label matrix fikrinin, bir molek?l i?in birden fazla etiket ta??yabilme mant???yla nas?l ili?kili oldu?unu.
- Oyuncak ?rnek ile ger?ek proje datas? aras?ndaki fark?.


## RDKit bu projede ne yapar?

Projenin bilimsel hedefi ??C NMR spektrumlar?ndan fonksiyonel grup tahmini yapmakt?r. RDKit burada do?rudan "spektrumu anlayan" bir model de?ildir. RDKit daha ?ok molek?l yap?s? taraf?nda yard?mc? olur.

RDKit ile yap?labilecek haz?rl?k i?leri ?unlara benzer:

- SMILES veya benzeri yap? g?sterimlerini Python i?inde molek?l nesnesine ?evirmek.
- Molek?l?n atom, ba?, halka, aromatiklik gibi yap?sal ?zelliklerini incelemek.
- Basit descriptor'lar ve fingerprint'ler ?retmenin ne anlama geldi?ini ??renmek.
- SMARTS desenleri ile belirli alt yap?lar? aramay? kavramak.
- Fonksiyonel grup etiketlerinin ileride nas?l bir label matrix'e d?n??ebilece?ini d???nmek.

Bu notebook yaln?zca bu kavramlar? tan?t?r. Final label schema, veri kayna?? karar? veya modelleme karar? vermez.


In [ ]:
# RDKit bu ortamda kurulu olmayabilir.
# Bu y?zden import i?lemini g?venli yap?yoruz: RDKit yoksa notebook tamamen ??kmesin.

RDKit_AVAILABLE = False
rdkit_import_error = None

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    try:
        from rdkit.Chem import rdFingerprintGenerator
    except Exception:
        rdFingerprintGenerator = None
    RDKit_AVAILABLE = True
except Exception as exc:
    rdkit_import_error = exc

if RDKit_AVAILABLE:
    print("RDKit bulundu. Kod h?creleri ?al??t?r?labilir.")
else:
    print("RDKit bu ortamda bulunamad?.")
    print("Notebook kavramsal olarak okunabilir; RDKit gerektiren h?creler g?venli ?ekilde atlanacak.")
    print(f"Import hatas?: {type(rdkit_import_error).__name__}: {rdkit_import_error}")


## K???k oyuncak molek?l listesi

?lk notebooklarda ger?ek NMReDATA dosyas?n? ana veri olarak kullanm?yoruz. Bunun yerine k???k, kontrol edilebilir ve kolay yorumlanabilir molek?llerle ba?l?yoruz.

Bu liste final proje dataset'i de?ildir. Sadece RDKit kavramlar?n? g?rmek i?in se?ilmi? oyuncak ?rneklerdir.


In [ ]:
toy_molecules = [
    {"name": "Etanol", "smiles": "CCO", "note": "Basit alkol ?rne?i"},
    {"name": "Aseton", "smiles": "CC(=O)C", "note": "Basit karbonil ?rne?i"},
    {"name": "Asetik asit", "smiles": "CC(=O)O", "note": "Karboksilik asit ?rne?i"},
    {"name": "Benzen", "smiles": "c1ccccc1", "note": "Aromatik halka ?rne?i"},
    {"name": "Etilamin", "smiles": "CCN", "note": "Basit amin ?rne?i"},
]

def print_rows(rows, columns):
    """K???k ??retici tablolar? ek paket kullanmadan yazd?r?r."""
    widths = []
    for column in columns:
        values = [str(row.get(column, "")) for row in rows]
        widths.append(max([len(column)] + [len(value) for value in values]))

    header = " | ".join(column.ljust(width) for column, width in zip(columns, widths))
    line = "-+-".join("-" * width for width in widths)
    print(header)
    print(line)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(width) for column, width in zip(columns, widths)))

print_rows(toy_molecules, ["name", "smiles", "note"])


## SMILES ? Mol object

SMILES, molek?l yap?s?n? tek sat?rl?k metin olarak yazman?n yayg?n bir yoludur. RDKit bu metni okuyup bir `Mol` nesnesine ?evirebilir.

`Mol` nesnesi, molek?l? Python i?inde atomlar, ba?lar ve yap?sal ili?kilerle temsil eder. B?ylece metin olarak duran kimyasal bilgi, programla incelenebilir h?le gelir.


In [ ]:
parsed_molecules = []

if RDKit_AVAILABLE:
    for item in toy_molecules:
        mol = Chem.MolFromSmiles(item["smiles"])
        parsed_molecules.append({**item, "mol": mol})

    rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        rows.append({
            "name": item["name"],
            "input_smiles": item["smiles"],
            "canonical_smiles": Chem.MolToSmiles(mol) if mol is not None else "OKUNAMADI",
            "atom_count": mol.GetNumAtoms() if mol is not None else "-",
            "bond_count": mol.GetNumBonds() if mol is not None else "-",
        })

    print_rows(rows, ["name", "input_smiles", "canonical_smiles", "atom_count", "bond_count"])
else:
    print("RDKit olmad??? i?in SMILES -> Mol d?n???m? bu ortamda ?al??t?r?lmad?.")
    print("Kavramsal fikir: SMILES metni, RDKit i?inde programla incelenebilir bir Mol nesnesine d?n???r.")


## Descriptor nedir?

Descriptor, molek?l?n say?sal veya kategorik bir ?zetidir. ?rne?in molek?l a??rl???, atom say?s?, halka say?s? veya hidrojen ba?? verici/al?c? say?s? birer descriptor olabilir.

Bu notebookta descriptor'lar yaln?zca kavram olarak g?sterilir. Bu de?erler model girdisi olarak kullan?lmaz, kar??la?t?rmal? sonu? veya benchmark ?retilmez.


In [ ]:
if RDKit_AVAILABLE:
    descriptor_rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        descriptor_rows.append({
            "name": item["name"],
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "mol_weight": round(Descriptors.MolWt(mol), 2),
            "h_donor": rdMolDescriptors.CalcNumHBD(mol),
            "h_acceptor": rdMolDescriptors.CalcNumHBA(mol),
        })

    print_rows(descriptor_rows, ["name", "formula", "mol_weight", "h_donor", "h_acceptor"])
else:
    print("RDKit olmad??? i?in descriptor ?rne?i ?al??t?r?lmad?.")


## Fingerprint nedir?

Fingerprint, molek?l yap?s?n? sabit uzunlukta bit dizisi gibi temsil etmeye yarayan yap?sal bir ?zettir. Bir bitin a??k olmas?, belirli bir yerel yap? par?as?n?n molek?lde bulundu?unu g?sterebilir.

Fingerprint kavram? ileride makine ??renmesiyle ili?kilendirilebilir; fakat bu notebookta model kurulmaz ve performans skoru ?retilmez. Burada yaln?zca "molek?l yap?s? say?sal temsile nas?l yakla??r?" sorusuna giri? yap?yoruz.


In [ ]:
if RDKit_AVAILABLE and rdFingerprintGenerator is not None:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=128)
    fingerprint_rows = []

    for item in parsed_molecules:
        mol = item["mol"]
        fingerprint = generator.GetFingerprint(mol)
        on_bits = list(fingerprint.GetOnBits())
        fingerprint_rows.append({
            "name": item["name"],
            "on_bit_count": len(on_bits),
            "first_on_bits": on_bits[:8],
        })

    print_rows(fingerprint_rows, ["name", "on_bit_count", "first_on_bits"])
elif RDKit_AVAILABLE:
    print("Bu RDKit kurulumunda rdFingerprintGenerator bulunamad?; fingerprint ?rne?i atland?.")
else:
    print("RDKit olmad??? i?in fingerprint ?rne?i ?al??t?r?lmad?.")


## SMARTS nedir?

SMARTS, molek?l i?inde belirli alt yap?lar? aramak i?in kullan?lan desen dilidir. SMILES bir molek?l? yazmaya yararken, SMARTS bir yap? motifini aramaya yarar.

?rne?in alkol, karbonil, karboksilik asit veya aromatik halka gibi basit desenler SMARTS ile aranabilir. A?a??daki desenler ??retici oyuncak ?rneklerdir; final fonksiyonel grup label schema de?ildir.


In [ ]:
toy_smarts_patterns = [
    {"label": "oyuncak_alkol", "smarts": "[OX2H]"},
    {"label": "oyuncak_karbonil", "smarts": "[CX3]=[OX1]"},
    {"label": "oyuncak_karboksilik_asit", "smarts": "C(=O)[OX2H1]"},
    {"label": "oyuncak_aromatik_halka", "smarts": "c1ccccc1"},
    {"label": "oyuncak_amin", "smarts": "[NX3]"},
]

if RDKit_AVAILABLE:
    pattern_rows = []
    for pattern in toy_smarts_patterns:
        smarts_mol = Chem.MolFromSmarts(pattern["smarts"])
        pattern_rows.append({
            "label": pattern["label"],
            "smarts": pattern["smarts"],
            "valid": smarts_mol is not None,
        })

    print_rows(pattern_rows, ["label", "smarts", "valid"])
else:
    print("RDKit olmad??? i?in SMARTS desenleri do?rulanmad?.")
    print_rows(toy_smarts_patterns, ["label", "smarts"])


## Label matrix fikri

Fonksiyonel grup tahmini ?ok etiketli (multi-label) bir problem olarak d???n?l?r: bir molek?lde ayn? anda birden fazla fonksiyonel grup bulunabilir.

Bu fikir tablo h?linde ??yle temsil edilebilir: sat?rlar molek?ller, s?tunlar etiketlerdir. H?credeki `1`, ilgili etiketin oyuncak kurala g?re bulundu?unu; `0`, bulunmad???n? g?sterir.

A?a??daki tablo sadece ??retici bir oyuncak label matrix'tir. Final proje label schema de?ildir, model girdisi de?ildir ve bilimsel sonu? de?ildir.


In [ ]:
if RDKit_AVAILABLE:
    compiled_patterns = []
    for pattern in toy_smarts_patterns:
        compiled_patterns.append({
            "label": pattern["label"],
            "query": Chem.MolFromSmarts(pattern["smarts"]),
        })

    matrix_rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        row = {"name": item["name"]}
        for pattern in compiled_patterns:
            query = pattern["query"]
            row[pattern["label"]] = int(query is not None and mol.HasSubstructMatch(query))
        matrix_rows.append(row)

    matrix_columns = ["name"] + [pattern["label"] for pattern in toy_smarts_patterns]
    print_rows(matrix_rows, matrix_columns)
else:
    print("RDKit olmad??? i?in oyuncak label matrix otomatik ?retilmedi.")
    print("Kavramsal fikir: sat?rlar molek?ller, s?tunlar fonksiyonel grup etiketleri olur.")


## ??C NMR projesiyle ba?lant?

Bu projenin ana y?n? ??C NMR spektrumlar?ndan fonksiyonel grup tahminidir. RDKit'in burada ??retti?i taraf, molek?l yap?s?n? g?venli ve denetlenebilir bi?imde ele almakt?r.

?leride RDKit ?u t?r d???nme ad?mlar?nda yard?mc? olabilir:

- Molek?l yap?s?ndan fonksiyonel grup var/yok etiketlerinin nas?l t?retilebilece?ini anlamak.
- Yak?n veya duplicate molek?l risklerini tart??mak.
- Kimyasal yap?n?n metadata ve spektrum bilgisiyle nas?l ili?kilendirilebilece?ini g?rmek.
- Notebooklarda hata toleransl? okuma ve kalite kontrol al??kanl??? kazanmak.

Bu notebookta `nmrshiftdb2rawdata.nmredata.sd` dosyas? ana veri olarak kullan?lmaz. O dosya yaln?zca ileride, daha ger?ek?i SDF/NMReDATA okuma ve debugging ??renimi i?in `learning sample / pilot example` stat?s?nde kullan?lacakt?r.


## Bu notebookta bilin?li olarak yap?lmayanlar

- Ger?ek veri indirme yap?lmad?.
- Final dataset se?imi yap?lmad?.
- `data/raw/` veya `data/processed/` kullan?lmad?.
- Model kurulmad?.
- Train/test split yap?lmad?.
- Macro F1, accuracy, benchmark veya skor ?retilmedi.
- ADR kabul? veya learning gate ge?i?i yap?lmad?.

Bu s?n?rlar, Command Center'?n g?venli ??renme ve haz?rl?k rol?n? korumak i?indir.


In [ ]:
notebook_boundary_check = {
    "uses_toy_molecules_only": True,
    "downloads_data": False,
    "trains_model": False,
    "creates_train_test_split": False,
    "computes_metrics_or_benchmark": False,
    "selects_final_dataset": False,
}

for key, value in notebook_boundary_check.items():
    print(f"{key}: {value}")


## Kapan??

RDKit'i bu a?amada bir modelleme arac? gibi de?il, molek?l yap?s?n? anlamay? ve denetlenebilir kimyasal veri haz?rl??? d???ncesini destekleyen bir ara? gibi okumak en sa?l?kl? ba?lang??t?r.

Bir sonraki ad?mda ama?, basit molek?l temsillerini daha rahat okumak ve RDKit nesnelerinin hangi bilgileri ta??d???n? daha yak?ndan incelemek olabilir.
